# Create Ludo Game Dataset

## Overview
This notebook generates a synthetic Ludo game dataset for training a machine learning model to predict winning combinations.

## What Was Done
- Simulated **1,000+ Ludo games** for 4 players: **Red, Green, Yellow, and Blue**
- Each game follows standard Ludo rules:
  - Players start at position **0** and must reach position **52** to win
  - Dice rolls are random integers between **1 and 6**
  - If a player overshoots position 52, they bounce back
  - Rolling a **6** grants the player an extra turn
- Simulation continues until at least **10,000 rows** of data are collected
- Each row records: `Game_ID`, `Turn`, `Player`, `Dice_Roll`, `Position`, `Is_Winner`
- The `Is_Winner` column is set to **1** for all of the winning player's moves in a game, and **0** for all others
- The final dataset is saved as `ludo_dataset.csv` in the `data file/Raw_Data/` directory

## Output
| Column | Description |
|---|---|
| `Game_ID` | Unique identifier for each simulated game |
| `Turn` | Turn number within the game |
| `Player` | Player colour (Red, Green, Yellow, Blue) |
| `Dice_Roll` | Value rolled on the dice (1–6) |
| `Position` | Player's board position after the move (0–52) |
| `Is_Winner` | 1 if this player won the game, 0 otherwise |


In [ ]:
# Creating a Ludo game dataset for training a model to predict winning combinations
import random
import pandas as pd
import os

def simulate_ludo(min_rows=10000):
    all_records = []
    players = ['Red', 'Green', 'Yellow', 'Blue']
    game_id = 0

    while len(all_records) < min_rows:
        positions = {player: 0 for player in players}  # All players start at position 0
        turn_number = 0
        winner = None

        while winner is None:
            for player in players:
                dice_roll = random.randint(1, 6)
                turn_number += 1

                # Move piece according to dice roll (only if not already home)
                if positions[player] < 52:
                    positions[player] += dice_roll
                    if positions[player] > 52:
                        # Bounce back if overshooting position 52
                        positions[player] = 52 - (positions[player] - 52)

                # Record this turn
                all_records.append({
                    'Game_ID': game_id,
                    'Turn': turn_number,
                    'Player': player,
                    'Dice_Roll': dice_roll,
                    'Position': positions[player],
                    'Is_Winner': 0  # placeholder, updated after game ends
                })

                # Check winning condition
                if positions[player] == 52:
                    winner = player
                    break

                # Extra turn if dice roll is 6
                if dice_roll == 6:
                    continue

            if winner:
                break

        # Mark the winner's moves in the records for this game
        for record in all_records:
            if record['Game_ID'] == game_id and record['Player'] == winner:
                record['Is_Winner'] = 1

        game_id += 1

    return all_records

# Generate the dataset (stops once at least 10,000 rows are collected)
ludo_records = simulate_ludo(min_rows=10000)

# Convert to DataFrame
df = pd.DataFrame(ludo_records)

# Save to CSV in the Raw_Data folder
save_path = os.path.join('..', 'data file', 'Raw_Data', 'ludo_dataset.csv')
df.to_csv(save_path, index=False)

print(f"Dataset created with {len(df)} rows across {df['Game_ID'].nunique()} games")
print(f"Saved to: {os.path.abspath(save_path)}")
print(df.head(10))
print("\nWinner distribution:")
winners = df[df['Is_Winner'] == 1].groupby('Player').size()
print(winners)


Dataset created with 10021 rows across 173 games, saved as 'ludo_dataset.csv'
   Game_ID  Turn  Player  Dice_Roll  Position  Is_Winner
0        0     1     Red          1         1          0
1        0     2   Green          5         5          0
2        0     3  Yellow          4         4          0
3        0     4    Blue          6         6          1
4        0     5     Red          1         2          0
5        0     6   Green          4         9          0
6        0     7  Yellow          3         7          0
7        0     8    Blue          6        12          1
8        0     9     Red          2         4          0
9        0    10   Green          2        11          0

Winner distribution:
Player
Blue      651
Green     694
Red       773
Yellow    457
dtype: int64
